In [1]:
#!pip install pyspark==3.5

In [2]:
#!pip install -U py4j

In [3]:
import pandas as pd

In [5]:
historical_data = pd.read_csv("files/all_except_last_orders.csv")
last_orders_data = pd.read_csv("files/last_orders_subset.csv")

In [6]:
historical_data['quantity'] = 1

In [7]:
historical_data.head()
historical_data.drop(["Order", "Delivery Date", "Name"], axis=1, inplace=True)

In [8]:
import sys
import os
import numpy as np
from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType, StringType

def als_factorize_numpy(R, num_factors=3, reg_param=0.1, max_iter=20):
    # Initialize Spark Session
    spark = SparkSession.builder \
        .appName("ALS Matrix Factorization") \
        .getOrCreate()

    # Convert Pandas DataFrame to Spark DataFrame
    # Explicit schema is safer
    schema = StructType([
        StructField("user", IntegerType(), True),
        StructField("item", IntegerType(), True),
        StructField("rating", FloatType(), True)
    ])

    # Ensure input is a Pandas DataFrame with correct types
    if not isinstance(R, pd.DataFrame):
        # If it's a numpy array or sparse matrix, we need to convert it
        # But based on usage, it seems to expect a DataFrame with columns 0, 1, 2
        pass

    ratings_df = spark.createDataFrame(R, schema=schema)

    # Train ALS Model
    # checkpointInterval=-1 disables checkpointing, which prevents "HADOOP_HOME not set" errors on Windows
    als = ALS(
        maxIter=max_iter,
        regParam=reg_param,
        userCol="user",
        itemCol="item",
        ratingCol="rating",
        coldStartStrategy="drop",
        implicitPrefs=True,
        nonnegative=True,
        rank=num_factors,
        checkpointInterval=-1
    )

    model = als.fit(ratings_df)

    # Get User and Item Factors
    # SORTING IS CRITICAL: Spark does not guarantee order.
    # We must sort by ID to ensure the matrix rows correspond to sorted SKU IDs.
    user_factors = model.userFactors.toPandas().sort_values("id")
    item_factors = model.itemFactors.toPandas().sort_values("id")

    # Extract matrices
    W = np.array(user_factors['features'].tolist())
    H = np.array(item_factors['features'].tolist())

    # Calculate MSE
    predictions = model.transform(ratings_df)
    evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")
    rmse = evaluator.evaluate(predictions)
    mse = rmse ** 2

    spark.stop()

    return W, H, mse

In [9]:
import os
import sys

# Ensure correct data types for PySpark DataFrame creation
ratings_data = historical_data[['Member', 'SKU', 'quantity']].copy()

usr_to_int = {usr: idx for idx, usr in enumerate(ratings_data['Member'].unique())}

ratings_data['Member'] = ratings_data['Member'].map(usr_to_int)
ratings_data['SKU'] = ratings_data['SKU'].astype(int)
ratings_data['quantity'] = ratings_data['quantity'].astype(float)

num_factors = 15
reg_param = 0.01
max_iter = 10

# Call the function with the prepared data and environment
W, H, mse = als_factorize_numpy(ratings_data, num_factors, reg_param, max_iter)

print("W (Order Feature Matrix):")
print(W)
print("\nH (SKU Feature Matrix):")
print(H)
print("--------------------------------------------------------------")
print(f"Final MSE = {mse:.4f}")
print("--------------------------------------------------------------")

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [ ]:
from sklearn.metrics import pairwise_distances
from scipy.spatial.distance import cosine, correlation

# Calculate Cosine Similarity between Item Factors
sku_sim = 1 - pairwise_distances(H, metric="cosine")

# Create DataFrame with correct SKU IDs as index/columns
# Since we sorted item_factors by ID in the function, H corresponds to sorted unique SKUs
# We use the unique SKUs from the training data to label the matrix
unique_skus = sorted(ratings_data['SKU'].unique())
sku_sim_df = pd.DataFrame(sku_sim, index=unique_skus, columns=unique_skus)

In [ ]:
sku_sim_df

,6884195,7541573,7543241,7547271,7547296,7547323,7548497,7548498,7548511,7548730,...,93141092,93141093,93156751,93174226,93176429,93176430,93176431,93289485,93289486,93289487
6884195,1.000000,0.310882,0.048955,0.083252,0.536678,0.103315,0.071426,0.239508,0.142498,0.117910,...,0.232380,0.113997,0.033765,0.058311,0.176157,0.188347,0.116807,0.000000,0.050717,0.281462
7541573,0.310882,1.000000,0.319368,0.697166,0.199749,0.708512,0.504100,0.613913,0.616228,0.317563,...,0.194477,0.018093,0.405360,0.352500,0.296240,0.395411,0.122946,0.206298,0.416389,0.304350
7543241,0.048955,0.319368,1.000000,0.298975,0.200690,0.000000,0.140274,0.172800,0.286141,0.550060,...,0.278048,0.304108,0.080070,0.102863,0.206222,0.239399,0.144158,0.187464,0.568393,0.049924
7547271,0.083252,0.697166,0.298975,1.000000,0.395848,0.711269,0.360988,0.807155,0.549867,0.378947,...,0.254333,0.043603,0.520571,0.161153,0.326556,0.480077,0.099651,0.000347,0.486402,0.124824
7547296,0.536678,0.199749,0.200690,0.395848,1.000000,0.264650,0.050408,0.530053,0.197627,0.406941,...,0.183118,0.089340,0.229456,0.086774,0.242304,0.232644,0.000000,0.000000,0.352144,0.194151
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93176430,0.188347,0.395411,0.239399,0.480077,0.232644,0.577755,0.537364,0.323335,0.573420,0.320131,...,0.332806,0.148165,0.448005,0.475455,0.312813,1.000000,0.554039,0.553119,0.287211,0.169132
93176431,0.116807,0.122946,0.144158,0.099651,0.000000,0.140989,0.207369,0.000000,0.337029,0.085370,...,0.614045,0.486159,0.092900,0.386775,0.488912,0.554039,1.000000,0.278407,0.046365,0.058160
93289485,0.000000,0.206298,0.187464,0.000347,0.000000,0.336978,0.586924,0.048714,0.467424,0.000000,...,0.034907,0.137883,0.512544,0.732844,0.000000,0.553119,0.278407,1.000000,0.316356,0.572335
93289486,0.050717,0.416389,0.568393,0.486402,0.352144,0.030296,0.466776,0.102857,0.061273,0.635016,...,0.296780,0.035050,0.282184,0.547095,0.267520,0.287211,0.046365,0.316356,1.000000,0.487139


In [ ]:
last_orders_data.head()
grouped_last_df = last_orders_data.groupby('Order')['SKU'].apply(list).reset_index()
grouped_last_df.head()

,Order,SKU
0,7341985,"[34987567, 15668455, 15669869, 15669860, 15668..."
1,7344710,"[15668462, 15669825, 15669772, 15668467, 15668..."
2,7345164,"[92435740, 7625765, 34986328, 15668451, 768935..."
3,7347668,"[15669817, 15668378, 15669874, 15669777, 15669..."
4,7348817,"[7580855, 21408952, 15669829, 15670267, 156697..."


In [ ]:
# for each order in last orders, find top 5 recommended SKUs based on similarity
def recommend_skus(order_skus, sku_sim_df, top_n=5):
    """
    Recommend SKUs by collecting all similar items from basket SKUs,
    sorting by their maximum similarity, and taking top N unique items.
    
    Args:
        order_skus: List of SKUs in the current order
        sku_sim_df: SKU similarity matrix
        top_n: Number of recommendations to return
    """
    # Track the maximum similarity score for each candidate SKU
    candidate_max_sim = {}
    
    for sku in order_skus:
        if sku in sku_sim_df.index:
            # Get all similarities for this SKU (excluding itself)
            similarities = sku_sim_df.loc[sku].drop(sku, errors='ignore')
            
            for candidate_sku, similarity in similarities.items():
                # Keep the maximum similarity score across all basket items
                candidate_max_sim[candidate_sku] = max(
                    candidate_max_sim.get(candidate_sku, 0), 
                    similarity
                )
    
    # Remove SKUs already in the order
    for sku in order_skus:
        candidate_max_sim.pop(sku, None)
    
    # Sort by similarity (descending) and take top N
    recommended_skus = sorted(
        candidate_max_sim.keys(),
        key=lambda x: candidate_max_sim[x],
        reverse=True
    )[:top_n]
    
    return recommended_skus

grouped_last_df['Recommended_SKUs'] = grouped_last_df['SKU'].apply(lambda skus: recommend_skus(skus, sku_sim_df, top_n=5))
grouped_last_df.head()

,Order,SKU,Recommended_SKUs
0,7341985,"[34987567, 15668455, 15669869, 15669860, 15668...","[15669780, 15669777, 15669880, 15669821, 15669..."
1,7344710,"[15668462, 15669825, 15669772, 15668467, 15668...","[15668688, 15668451, 92436884, 15669910, 15668..."
2,7345164,"[92435740, 7625765, 34986328, 15668451, 768935...","[92286348, 7594820, 15668453, 7621702, 7579697]"
3,7347668,"[15669817, 15668378, 15669874, 15669777, 15669...","[15669771, 15669815, 7580823, 7624792, 15669799]"
4,7348817,"[7580855, 21408952, 15669829, 15670267, 156697...","[15669861, 15669884, 7580811, 7628087, 15669812]"


In [ ]:
# Create a mapping of Order to Member from the original last_orders_data
order_member_map = last_orders_data[['Order', 'Member']].drop_duplicates()

# Prepare the submission DataFrame
# Explode the list of recommended SKUs into separate rows
submission_df = grouped_last_df[['Order', 'Recommended_SKUs']].explode('Recommended_SKUs')

# Rename columns to match requirements
submission_df = submission_df.rename(columns={'Recommended_SKUs': 'SKU'})

# Merge to get the Member column
submission_df = submission_df.merge(order_member_map, on='Order', how='left')

# Add a sequential ID column
submission_df.reset_index(drop=True, inplace=True)
submission_df['ID'] = submission_df.index + 1

# Reorder columns as requested
submission_df = submission_df[['ID', 'Order', 'SKU', 'Member']]

# Save to CSV
submission_df.to_csv('submission_matrix.csv', index=False)

# Display the first few rows to verify
submission_df.head()

,ID,Order,SKU,Member
0,1,7341985,15669780,SWCCWNZ
1,2,7341985,15669777,SWCCWNZ
2,3,7341985,15669880,SWCCWNZ
3,4,7341985,15669821,SWCCWNZ
4,5,7341985,15669884,SWCCWNZ
